In [0]:
# Optimized Join Job Design — Databricks PySpark

## Problem Context
We are joining two large tables:
- Orders fact table (~30M rows)
- Customers dimension table (~20M rows)

Goal → Fast, scalable, production-grade join pipeline.

---

## 1. Storage Format
Always store source tables in **Delta format** because it provides:
- Predicate pushdown
- Column pruning
- File skipping
- Statistics for optimizer
- ACID guarantees

---

## 2. Join Strategy Decision
Join type must be chosen based on data size:

| Scenario | Strategy |
|--------|----------|
Small dimension | Broadcast Join |
Both large | Shuffle Join |
Skewed keys | Salted Join |

For this case → both large → **Shuffle Join**

---

## 3. Partitioning Strategy
Partition only if queries filter by that column.

Good:
- order_date
- region

Bad:
- customer_id (high cardinality)

---

## 4. Cluster Configuration Guidelines

Recommended cluster config:

- Runtime → Latest LTS
- Photon → Enabled
- Workers → 4–8
- Autoscaling → Enabled
- Worker type → Memory optimized

Why?
Joins are memory + shuffle heavy.

---

## 5. Performance Principles

Always follow:

✔ Column pruning  
✔ Predicate filtering early  
✔ Avoid wide transformations early  
✔ Avoid unnecessary shuffles  
✔ Cache only reused DataFrames  

---

## 6. Data Skew Detection
Check skew before joining:

Symptoms:
- One task runs much longer
- Executor memory spikes
- Shuffle spill increases

---

## 7. Adaptive Query Execution (Must Enable)

AQE automatically:
- fixes skew joins
- optimizes partition size
- changes join strategy dynamically

---

## 8. Delta Optimization After Write

Run optimize for faster reads:

OPTIMIZE table_name ZORDER BY (join_key)

---

## 9. Production Job Settings

Recommended job configuration:

Retries → 2  
Timeout → 2h  
Max concurrent runs → 1  
Alerts → Enabled  

---

## 10. Monitoring Checklist

Check Spark UI → SQL tab:

| Issue | Indicator |
|------|-----------|
Skew | One task extremely slow |
Shuffle overload | Spill to disk |
Bad partitioning | Too many tiny tasks |

---

## Golden Rule
Performance mostly depends on:
DATA LAYOUT + PARTITIONING + CLUSTER CONFIG

Not just code.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# -------------------------
# Enable Adaptive Query Execution
# -------------------------
spark.conf.set("spark.sql.adaptive.enabled", "true")

# -------------------------
# Read Tables (Column Pruning)
# -------------------------
orders = (
    spark.read.format("delta")
    .table("sales.orders")
    .select("order_id","customer_id","amount","order_date")
)

customers = (
    spark.read.format("delta")
    .table("dim.customers")
    .select("customer_id","country","segment")
)

# -------------------------
# Repartition Before Join
# Helps distribute shuffle evenly
# -------------------------
orders = orders.repartition("customer_id")
customers = customers.repartition("customer_id")

# -------------------------
# Join Tables
# -------------------------
joined_df = (
    orders.join(
        customers,
        on="customer_id",
        how="inner"
    )
)

# -------------------------
# Write Output (Optimized)
# -------------------------
(joined_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema","true")
    .saveAsTable("gold.orders_enriched")
)


In [0]:
# -------------------------
# Check Data Skew
# -------------------------
orders.groupBy("customer_id") \
      .count() \
      .orderBy(F.desc("count")) \
      .show(10)

# -------------------------
# If Dimension becomes small after filter
# Use Broadcast Join
# -------------------------
filtered_customers = customers.filter("country = 'US'")

joined_df = orders.join(
    broadcast(filtered_customers),
    "customer_id"
)

# -------------------------
# Explain Execution Plan
# -------------------------
joined_df.explain(True)

# -------------------------
# Optimize Delta Table After Write
# Run in SQL cell
# -------------------------
# OPTIMIZE gold.orders_enriched
# ZORDER BY (customer_id)
